# 01 · The embedding engine

The core: `numpy` and `rdkit`, nothing else. Three verbs.

- `embed(spec, fix=, constrain=, n=)` returns `Conformers`
- `minimize(spec, fix=, constrain=)` relaxes a geometry you already have, no search
- `enumerate_isomers(mol, geometry)` gives the coordination arrangements of a metal

`rxembed.core.embed` takes a Mol with explicit hydrogens or a metal Isomer, never a string.


In [ ]:
from rdkit import Chem

import rxembed.core as rx

rx.set_verbose("INFO")  # stage summaries; use DEBUG for per-conformer diagnostics

mol = Chem.AddHs(Chem.MolFromSmiles("CCCCO"))  # atoms C0 C1 C2 C3 O4

## A free embed

`n=` is the conformer count. `seed=` is always set, so a run is reproducible; RDKit's own default draws
from the global RNG and makes every result depend on what ran before it.


In [ ]:
confs = rx.embed(mol, n=8).minimize()
print(len(confs), "conformers")
print({i: round(e, 2) for i, e in confs.energies.items()})

## Constrain: state a target

A scalar distance target becomes a ±0.1 Å window. That window biases the RDKit embed and is applied again
during the restrained-UFF relax. Keys are 0-based atom indices; name them before building the constraint.


In [ ]:
terminal_c, _, _, _, alcohol_o = rx.match(mol, "[CH3][CH2][CH2][CH2][OX2H]")
target = 2.8  # Å

soft = rx.embed(mol, constrain={(terminal_c, alcohol_o): target}, n=8).minimize()
print("target 2.8 ->", {k: round(v, 2) for k, v in soft.measure((terminal_c, alcohol_o)).items()})

## Fix: a number you mean

A `fix` stated as a number is a tight-window UFF pull, not a snap. Read it back with `.measure()`.
A `fix` stated as coordinates is different: those atoms are Kabsch-grafted and restored exactly.


In [ ]:
held = rx.embed(mol, fix={(terminal_c, alcohol_o): 3.0}, n=8).minimize()
print("asked 3.0 ->", {k: round(v, 2) for k, v in held.measure((terminal_c, alcohol_o)).items()})

ang = rx.embed(mol, fix={(0, 1, 2): 118.0}, n=6).minimize()
print("asked 118 deg ->", {k: round(v, 1) for k, v in ang.measure((0, 1, 2)).items()})

## A rigid core, held exactly

Give `fix` a list of indices and those atoms keep the coordinates they came in with. The molecule needs a
conformer for that, and the rest of it is still conformer-searched.


In [ ]:
from rdkit.Chem import rdDistGeom

ref = Chem.AddHs(Chem.MolFromSmiles("CC(=O)Nc1ccccc1"))
rdDistGeom.EmbedMolecule(ref, randomSeed=1)  # the geometry whose core we want to keep
core = [0, 1, 2, 3]  # the amide C-C(=O)-N

grafted = rx.embed(ref, fix=core, n=6).minimize()
print(f"{len(grafted.ids)} conformers with the four-atom amide core fixed")

## Relax something you already have

`minimize` is the same vocabulary with no conformer search: one geometry in, one relaxed geometry out.


In [ ]:
pulled = rx.minimize(confs.mol, fix={(terminal_c, alcohol_o): 3.4})
print("pulled to 3.4 ->", {k: round(v, 2) for k, v in pulled.measure((terminal_c, alcohol_o)).items()})

## Getting the geometries out

`.mol` is a real RDKit Mol, `.xyz()` a multi-frame block, `.dump(path)` writes it.


In [ ]:
print(confs.xyz().splitlines()[0], "atoms in frame 1")
print("conformer ids:", list(confs.ids))

## Next

- [`02_constraints`](02_constraints.ipynb) distance, angle and plane windows
- [`04_organic_ts`](04_organic_ts.ipynb) a reacting core from an .xyz
- [`05_templated_ts`](05_templated_ts.ipynb) a template on a fresh molecule
- [`08_energies`](08_energies.ipynb) workflow operations: search, dedup, real energies
